# Tutorial Notebook

This notebook is a practical walk-through for the **amortized Bayesian workflow** introduced in the paper.

Core idea: train one amortized posterior estimator/surrogate once, then reuse it across many datasets and escalate only when diagnostics require it.

In brief, the main steps are:

- Training phase: train an amortized estimator and validate it with simulated data.
- Inference phase (Step 1): fast amortized draws for real data + out-of-distribution (OOD) diagnostic for flagging datasets where the amortized draws may be unreliable
- Inference phase (Step 2): Pareto-smoothing importance sampling correction + Pareto-$\hat{k}$ diagnostic
- Inference phase (Step 3): MCMC fallback with amortized initializations

This notebook runs each part manually so you can inspect intermediate outputs and reuse the same pattern for your own tasks.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"
import numpy as np
import bayesflow as bf
import matplotlib.pyplot as plt

from amortized_bayesian_workflow import (
    ArtifactLayout,
    InferenceConfig,
    InferenceRunner,
)
from amortized_bayesian_workflow.approximators import (
    BayesFlowAmortizedPosterior,
)
from amortized_bayesian_workflow.backends import run_mcmc
from amortized_bayesian_workflow.psis import (
    compute_psis,
    resample_with_weights,
)
from amortized_bayesian_workflow import configure_logging

configure_logging("WARNING")
print(f"BayesFlow version: {bf.__version__}")

## Training phase: train an amortized estimator

An amortized posterior estimator takes in observed data and outputs an approximation to the posterior distribution. It can be trained with simulated data from the probabilistic model (prior predictive distribution), which is also known as **simulation-based inference (SBI)** in the literature. Here we use [BayesFlow](https://github.com/bayesflow-org/bayesflow) package for this purpose, which provide convenient APIs for building, training and validating amortized estimators.


Here we use the psychometric curve fitting example task from the paper. This task was implemented by wrapping a PyMC model into a format compatible with BayesFlow for amortized inference. One can also define a task by providing a `simulate_prior_predictive` function for prior predictive simulation.


In [ ]:
from amortized_bayesian_workflow.tasks.examples import (
    PsychometricTask,
)
from pathlib import Path

task = PsychometricTask(overdispersion=True)

layout = ArtifactLayout(
    root=Path("./artifacts"),
    task_name=task.task_name,
)
layout.ensure()

In [ ]:
from bayesflow.workflows import BasicWorkflow
from bayesflow.diagnostics import plots as bf_plots

import keras

num_train_simulations = 10000
num_validation_simulations = 500
batch_size = 256
epochs = 3
seed = 2026

keras.utils.set_random_seed(seed)

adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .rename("parameters", "inference_variables")
    .rename("observables", "summary_variables")
)
inference_network = bf.networks.CouplingFlow()
summary_network = bf.networks.DeepSet()
basic_amortized_workflow = BasicWorkflow(
    adapter=adapter,
    inference_network=inference_network,
    summary_network=summary_network,
    checkpoint_filepath=layout.models_dir,
    checkpoint_name="best_model",
    save_best_only=True,
)

train_sims = task.simulate_prior_predictive(num_train_simulations, seed=seed)
val_sims = task.simulate_prior_predictive(
    num_validation_simulations, seed=seed + 1
)

history = basic_amortized_workflow.fit_offline(
    data=train_sims,
    validation_data=val_sims,
    epochs=epochs,
    batch_size=batch_size,
)
_ = bf_plots.loss(history)

basic_amortized_workflow.approximator = keras.saving.load_model(
    layout.models_dir / "best_model.keras"
)

After training, we can validate the amortized estimator with simulated data. This validation step is important for checking the quality of the amortized estimator before applying it to real data.

In the paper, we recommend simulation-based calibration (SBC) and parameter recovery checking in particular. `BayesFlow` also provides other diagnostic plots such as coverage plots and posterior vairance contraction plots, which can be useful as well.

From the plots below, we can see that the trained amortized estimator is well-calibrated but some parameters are hard to recover, which does not necessarily indicate a problem with the amortized estimator as it can due to the inherent weak parameter identifiability given the limited observations.


In [ ]:
plot_fns = {
    "recovery": bf_plots.recovery,
    "calibration_ecdf": bf_plots.calibration_ecdf,
    "coverage": bf_plots.coverage,
    "z_score_contraction": bf_plots.z_score_contraction,
}

test_sims = task.simulate_prior_predictive(200)
post_draws = basic_amortized_workflow.approximator.sample(
    conditions=test_sims, num_samples=1000
)
# We transform to the original parameter space for better interpretability of the diagnostic plots, but this is optional.
estimates = task.transform_to_constrained_space(
    post_draws["parameters"].reshape(-1, post_draws["parameters"].shape[-1])
).reshape(post_draws["parameters"].shape)
targets = task.transform_to_constrained_space(test_sims["parameters"])

for k, plot_fn in plot_fns.items():
    _ = plot_fn(
        estimates=estimates,
        targets=targets,
        variable_names=task.var_names,
    )

In [ ]:
# Wrap the trained approximator and cache summary statistics of training data for OOD detection later.
approximator = BayesFlowAmortizedPosterior(
    basic_amortized_workflow.approximator
)
approximator.store_training_summaries(train_sims["observables"])
print(f"Summary statistics shape: {approximator.training_summaries.shape}")

# We can save the approximator to disk and load it for reuse in future.
approximator.save(layout.models_dir)

## Inference phase


Once we have a trained amortized estimator and validate it on simulated data, we can apply it to real data for inference.


In [ ]:
# Construct `InferenceRunner` with the trained amortized estimator.

config = InferenceConfig(seed=2026)
runner = InferenceRunner(task=task, approximator=approximator, config=config)

## Step 1: amortized posterior draws


In [ ]:
from amortized_bayesian_workflow.utils import read_from_file

# We read some real datasets from disk for testing.
test_observations = read_from_file(task.task_info_dir / "test_dataset.pkl")[
    :10
]
observation = test_observations[0]


In [ ]:
amortized = approximator.sample_and_log_prob(
    observation, num_samples=config.num_amortized_draws, seed=config.seed
)
q_samples = amortized.samples
log_q = amortized.log_prob
q_samples.shape, log_q.shape

## Step 1 diagnostic: OOD detection with Mahalanobis distance

The amortized estimator can provide fast posterior draws, but they may be unreliable if the real data is out-of-distribution (OOD) compared to the simulated data used for training. Therefore, we need diagnostics to flag such cases and proceed with PSIS corrections or MCMC sampling.

We use Mahalanobis distance on summary statistics to flag likely out-of-distribution datasets. Datasets passing this gate can accept amortized draws directly; rejected datasets proceed to Step 2.

It is important to note that the OOD diagnostic is not perfect and may have false positives or false negatives. For some applications, it may be desirable to proceed all datasets to Step 2 regardless of the OOD diagnostic. See the paper for more discussion on this point.


In [ ]:
runner._calibrate_mahalanobis_reference()

# all_summaries = runner._summary_statistics(test_observations)
# ood_results = runner._mahalanobis_reference.evaluate_batch(
#     all_summaries, alpha=runner.config.mahalanobis_alpha
# )

# mahalanobis_stats = np.array([r.statistic for r in ood_results], dtype=float)
# ood_rejected = np.array([r.rejected for r in ood_results], dtype=bool)
# cutoff = float(ood_results[0].cutoff) if len(ood_results) > 0 else np.nan

# mahalanobis_stats.shape, ood_rejected.sum(), cutoff

We can plot the OOD diagnostic results to inspect how many datasets are flagged as OOD and how they differ from Mahalanobis distances of the training data.


In [ ]:
runner._calibrate_mahalanobis_reference()

train_stats = runner._mahalanobis_reference.train_statistics
all_summaries = runner._summary_statistics(test_observations)
ood_results = runner._mahalanobis_reference.evaluate_batch(
    all_summaries, alpha=runner.config.mahalanobis_alpha
)
test_stats = np.array([r.statistic for r in ood_results], dtype=float)
cutoff = runner._mahalanobis_reference.threshold(
    alpha=runner.config.mahalanobis_alpha
)

bins = np.linspace(
    min(train_stats.min(), test_stats.min()),
    max(train_stats.max(), test_stats.max()),
    40,
)

plt.figure(figsize=(8, 5))
plt.hist(
    train_stats, bins=bins, density=True, alpha=0.5, label="Training data"
)
plt.hist(test_stats, bins=bins, density=True, alpha=0.6, label="Test data")
plt.axvline(
    cutoff,
    color="red",
    linestyle="--",
    linewidth=2,
    label=rf"Cutoff = {cutoff:.3f} ($\alpha$={runner.config.mahalanobis_alpha})",
)
plt.xlabel("Mahalanobis distance")
plt.ylabel("Density")
plt.title("Train vs Test Mahalanobis distances")
plt.legend()
plt.tight_layout()
plt.show()


## Step 2: Pareto-smoothed importance sampling (PSIS)

PSIS reweights amortized draws using the target posterior density.
The Pareto-$\hat{k}$ diagnostic tells whether corrected draws are reliable ($\hat{k}<0.7$)or should escalate to Step 3 ($\hat{k}>0.7$).


In [ ]:
log_post_vec = task.vectorized_log_posterior_fn(observation)
log_target = log_post_vec(q_samples)
psis = compute_psis(log_target=log_target, log_proposal=log_q)
psis.pareto_k, psis.ess

We can plot resample posterior draws after PSIS correction compare the corrected posterior samples with the original amortized draws to see how much correction is applied by PSIS.


In [ ]:
from amortized_bayesian_workflow.utils import corner_plot

posterior_draws_psis = resample_with_weights(
    samples=q_samples,
    weights=psis.smoothed_normalized_weights,
    num_draws=1000,
    replace=True,
)
_ = corner_plot(
    q_samples,
    posterior_draws_psis,
    labels=["amortized", "psis-corrected"],
    var_names=task.var_names,
    transform=task.transform_to_constrained_space,
)

## Step 3: MCMC fallback with amortized initializations

If PSIS is not reliable as indicated by the Pareto-$\hat{k}$ diagnostic, we run MCMC and initialize chains from amortized draws. This reuse strategy can reduce the warmup time and improve efficiency. Here we use the ChEES-HMC sampler as an example, which launches many short chains (e.g., 2048) in parallel and is designed to work with SIMD hardware like GPUs.


In [ ]:
backend_name = "blackjax_chees_hmc"
# backend_name = "blackjax_nuts"
# backend_name = "tfp_chees_hmc"
iter_warmup = 200
if backend_name in ["blackjax_chees_hmc", "tf_chees_hmc"]:
    options = {"num_superchains": 16, "subchains_per_superchain": 128}
    iter_sampling = 1
    initial_positions = q_samples[: options["num_superchains"]]
elif backend_name == "blackjax_nuts":
    options = {"num_chains": 4}
    iter_sampling = 500
    initial_positions = q_samples[: options["num_chains"]]

log_post_single = task.single_log_posterior_fn(
    observation
)  # Use a non-vectorized log-prob function for MCMC sampling, since most MCMC backends expect that.
mcmc_result = run_mcmc(
    backend_name=backend_name,
    initial_positions=initial_positions,
    log_prob_fn=log_post_single,
    iter_warmup=iter_warmup,
    iter_sampling=iter_sampling,
    options=options,
    seed=seed,
)
mcmc_result.backend, mcmc_result.draws.shape, mcmc_result.diagnostics

print("MCMC draws shape:", mcmc_result.draws.shape)
print(
    f"{mcmc_result.rhat_name}: {mcmc_result.diagnostics.get(mcmc_result.rhat_name)}"
)

In [ ]:
posterior_draws_mcmc = mcmc_result.draws.reshape(
    -1, mcmc_result.draws.shape[-1]
)
posterior_draws_mcmc.shape

---

## Run inference for many datasets on one call

This notebook serves as a reusable template for new tasks and datasets. We also provide a convenient `runner.run(...)` for demonstrating running the full inference phase for many datasets in one call. (In practice, you may want to use SLURM or other cluster computing systems for embarrassingly parallel runs of many datasets.)


In [ ]:
from pathlib import Path
from amortized_bayesian_workflow import (
    ArtifactLayout,
    InferenceConfig,
    InferenceRunner,
)
from amortized_bayesian_workflow.tasks.examples import PsychometricTask
from amortized_bayesian_workflow.utils import read_from_file
from amortized_bayesian_workflow.approximators import (
    BayesFlowAmortizedPosterior,
)

task = PsychometricTask()
layout = ArtifactLayout(
    root=Path("./artifacts"),
    task_name=task.task_name,
)
config = InferenceConfig(
    mcmc_backend="blackjax_nuts",
    seed=2026,
    rewrite_persisted_dataset_results=True,
    force_mcmc=True,
)

approximator = BayesFlowAmortizedPosterior.load(layout.models_dir)
runner = InferenceRunner(task=task, approximator=approximator, config=config)


test_observations = read_from_file(task.task_info_dir / "test_dataset.pkl")[
    :10
]
report = runner.run(test_observations[-2:])
report.summary_table()